In [1]:
%pip install earthengine-api pandas numpy matplotlib seaborn geopandas folium osmnx networkx

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import ee

In [3]:
ee.Authenticate()


Successfully saved authorization token.


In [3]:
ee.Initialize(project='project-0cf410a3-f35f-4911-9c5')

### Definitions
- What is CHRIPS?
    - Climate Hazards Group InfraRed Precipitation with Station data
    - It's a satellite derived daily rainfall dataset, across Africa and other regions.
- Spatial resolution: 0.05 degrees, meaning that each data point covers a 5km of estimation of rain fall.
- Temporal resolution(cadence): Daily (how data is updated)
- Period: 1981 - near present (2/3 weeks delay)


In [22]:
import datetime

SAFARI_BBOX = ee.Geometry.BBox(34.00, -4.50, 37.00, -1.50)

In [13]:
SAFARI_BBOX #Polygon representation for safari

ee.Geometry({
  "functionInvocationValue": {
    "functionName": "GeometryConstructors.Polygon",
    "arguments": {
      "coordinates": {
        "constantValue": [
          [
            [
              34.0,
              -1.5
            ],
            [
              34.0,
              -4.5
            ],
            [
              37.0,
              -4.5
            ],
            [
              37.0,
              -1.5
            ]
          ]
        ]
      },
      "geodesic": {
        "constantValue": false
      }
    }
  }
})

In [26]:
# total available data
# this maybe could be just a job running daily and updating some kind of DB?!

DATE_START = '2023-11-01'
DATE_END   = '2023-11-30'

testin_chirps = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
          .filterBounds(SAFARI_BBOX)
            .filterDate(DATE_START, DATE_END))

params = {
    'region': SAFARI_BBOX,
    'dimensions': 200,
    'framesPerSecond': 5,
    'min': 0,
    'max': 50,
    'palette': ['white', 'blue', 'green', 'yellow', 'red']
}

testin_chirps.getVideoThumbURL(params)

'https://earthengine.googleapis.com/v1/projects/project-0cf410a3-f35f-4911-9c5/videoThumbnails/36d38653bf79fd2812b2fdfb1da3fb36-705532f08e7cef115ff5cf7182dcc336:getPixels'

In [27]:
first_img = testin_chirps.first()
first_img.getInfo()

{'type': 'Image',
 'bands': [{'id': 'precipitation',
   'data_type': {'type': 'PixelType', 'precision': 'float'},
   'dimensions': [7200, 2000],
   'crs': 'EPSG:4326',
   'crs_transform': [0.05, 0, -180, 0, -0.05, 50]}],
 'version': 1747258142570294,
 'id': 'UCSB-CHG/CHIRPS/DAILY/20231101',
 'properties': {'system:time_start': 1698796800000,
  'system:footprint': {'type': 'LinearRing',
   'coordinates': [[-180, -90],
    [180, -90],
    [180, 90],
    [-180, 90],
    [-180, -90]]},
  'system:time_end': 1698883200000,
  'system:asset_size': 4681180,
  'system:index': '20231101'}}

In [21]:
#Filtering data

filtering_chirps = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
          .filterDate(DATE_START, DATE_END)
          .filterBounds(SAFARI_BBOX))

In [29]:
dates = filtering_chirps.aggregate_array('system:time_start').getInfo()

readable_dates = [
    datetime.datetime.fromtimestamp(d/1000).strftime('%Y-%m-%d')
    for d in dates
]

print(f'Date range: {readable_dates[0]} to {readable_dates[-1]}')
print(f'Days with data: {len(readable_dates)}')

Date range: 2023-11-01 to 2023-11-29
Days with data: 29


In [30]:
# Available data for the given date

dates = filtering_chirps.aggregate_array('system:time_start').getInfo()
readable_dates = [
    datetime.datetime.fromtimestamp(d/1000).strftime('%Y-%m-%d')
    for d in dates
]
print(f'Date range: {readable_dates[0]} to {readable_dates[-1]}')
print(f'Days with data: {len(readable_dates)}')

Date range: 2023-11-01 to 2023-11-29
Days with data: 29


In [33]:
# There's a missing day...
import pandas as pd

expected_dates = pd.date_range(
    start=DATE_START,
    end=DATE_END,
    freq='D'
).strftime('%Y-%m-%d').tolist()

missing = set(expected_dates) - set(readable_dates)

for d in sorted(missing):
    print(f"{d}")

2023-11-30


### Analyze the OSM vs CHIRPS

- We need to analyze if number_edges > number_pixes in a A->B route.
   - We need to figure it out how many roads (edges) fall within a single chirps pixels.

In [34]:
import osmnx as ox

G = ox.graph_from_place(
    "Tarangire National Park, Tanzania",
    network_type='all',
    retain_all=True
)

- We now check, concretely, how many road edges fall inside a single CHIRPS pixel.
- CHIRPS resolution is 0.05° (~5km).
- We build a grid of 0.05° cells over the same
bounding box we use for the road network, then spatially join road edges to grid cells.

In [38]:
import numpy as np
from shapely.geometry import box
import geopandas as gpd

CHIRPS_RESOLUTION_DEG = 0.05

nodes, edges = ox.graph_to_gdfs(G)

minx, miny, maxx, maxy = edges.total_bounds

minx = np.floor(minx / CHIRPS_RESOLUTION_DEG) * CHIRPS_RESOLUTION_DEG # Converting these OSM coordinates (real coordinates) to ranges of 0.05 degrees
miny = np.floor(miny / CHIRPS_RESOLUTION_DEG) * CHIRPS_RESOLUTION_DEG

#For example:
# (35.123, 12.33) -> (35.05, 12.25) (example)

maxx = np.ceil(maxx / CHIRPS_RESOLUTION_DEG) * CHIRPS_RESOLUTION_DEG
maxy = np.ceil(maxy / CHIRPS_RESOLUTION_DEG) * CHIRPS_RESOLUTION_DEG

xs = np.arange(minx, maxx, CHIRPS_RESOLUTION_DEG)
ys = np.arange(miny, maxy, CHIRPS_RESOLUTION_DEG)

In [40]:
init_minx, init_miny, init_maxx, init_maxy = edges.total_bounds

print(f'Initial mins; {init_minx}, {init_miny}, {init_maxx}, {init_maxy}')

print(f'Array of xs and ys: {xs}, {ys}')

pixels = []
for x in xs:
    for y in ys:
        pixels.append(box(x, y, x + CHIRPS_RESOLUTION_DEG, y + CHIRPS_RESOLUTION_DEG))

pixel_grid = gpd.GeoDataFrame(
    {"pixel_id": range(len(pixels))},
    geometry=pixels,
    crs=edges.crs,
)

print(f"CHIRPS pixels covering the park bbox: {len(pixel_grid)}")
print(f"Road edges in the park graph: {len(edges)}")

Initial mins; 35.8689897, -4.5263433, 36.303507, -3.680242
Array of xs and ys: [35.85 35.9  35.95 36.   36.05 36.1  36.15 36.2  36.25 36.3 ], [-4.55 -4.5  -4.45 -4.4  -4.35 -4.3  -4.25 -4.2  -4.15 -4.1  -4.05 -4.
 -3.95 -3.9  -3.85 -3.8  -3.75 -3.7 ]
CHIRPS pixels covering the park bbox: 180
Road edges in the park graph: 2439


In [13]:
edge_pixel_join = gpd.sjoin(
    edges.reset_index()[["u", "v", "key", "geometry"]],
    pixel_grid,
    how="inner",
    predicate="intersects",
)

edges_per_pixel = edge_pixel_join.groupby("pixel_id").size()
occupied_pixels = len(edges_per_pixel)

print(f"Pixels that actually contain at least one edge: {occupied_pixels}")
print(f"Edges per occupied pixel -> mean: {edges_per_pixel.mean():.1f}, "
      f"median: {edges_per_pixel.median():.0f}, max: {edges_per_pixel.max()}")
print(f"number_edges ({len(edges)}) > number_pixels ({occupied_pixels}): "
      f"{len(edges) > occupied_pixels}")

Pixels that actually contain at least one edge: 93
Edges per occupied pixel -> mean: 30.0, median: 8, max: 542
number_edges (2439) > number_pixels (93): True


**Finding:** `number_edges > number_pixels`

- many road segments share the same CHIRPS pixel.
- a single daily rainfall reading is therefore not "one value per road"
- this directly shapes how we integrate rainfall into edge risk later: we join edges to whichever pixel they intersect and copy that pixel's risk score onto every edge
in it, rather than expecting a 1:1 mapping.

**Is rainfall accumulative?**
Yes, for our purpose it has to be. A single day of light rain rarely makes a dirt track
impassable; several consecutive wet days do, because the ground stays saturated and
doesn't drain between events. Treating each day in isolation would under-weight the
scenario that actually matters (a multi-day storm), and over-weight an isolated light
shower. We'll use a **rolling 3-day accumulated precipitation** (sum of the current day
and the two prior days) as the signal that drives risk, recomputed per pixel per day.
Three days is a starting point — long enough to capture saturation building up, short
enough that risk decays again once the rain stops — and can be tuned once we have real
road-condition feedback.

**Realtime use / CHIRPS latency:**
CHIRPS is published with a 2-3 week delay, so it cannot drive live "is this road
passable right now" decisions. That's fine for this project's current scope, which is
route *planning* and risk modeling, not live dispatch: CHIRPS is well suited to
(a) building and calibrating the rainfall→risk relationship itself, and (b) giving a
seasonal/historical risk prior for a given route and month. If a live/real-time feature
is built later, we'd swap in a low-latency product (e.g. GPM IMERG, ~4h latency) behind
the same interface, since the risk model doesn't care where the accumulated-rainfall
number came from.

**Pixel → edge mapping:**
Confirmed above: `number_edges > number_pixels`. We assign each edge the risk of
whichever CHIRPS pixel its geometry intersects (spatial join, `predicate="intersects"`),
so every edge inside a pixel inherits that pixel's rainfall risk.

**Combining rainfall risk with road-type risk:**
The same rainfall should not affect a paved motorway and a dirt track equally. So the
final `surface_risk` isn't just the rainfall risk — it's the existing road-type base
risk (`HIGHWAY_SURFACE_RISK`) plus the rainfall risk scaled by how vulnerable that road
type is to rain, capped at 1.0:

```
surface_risk = min(1.0, base_road_risk + road_vulnerability[road_type] * rain_risk)
```

where paved/major roads get a low vulnerability multiplier and unpaved tracks get a
high one.